# Resultados de experimentos `speech`

Este notebook recorre las carpetas de `results` que empiezan por `speech`, lee el `final_results.json` de cada experimento, normaliza los datos, guarda una version consolidada y muestra los resultados de forma legible.

In [1]:
from pathlib import Path
import csv
import html
import json

from IPython.display import HTML, Markdown, display

RESULTS_DIR = Path.cwd()
if RESULTS_DIR.name != "results":
    RESULTS_DIR = Path("results")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"No se encontro la carpeta de resultados: {RESULTS_DIR.resolve()}")

RESULTS_DIR.resolve()

PosixPath('/home/odra/GitHub/ScraBrain/results')

In [2]:
def flatten_final_results(data: dict, experiment_dir: Path) -> dict:
    """Convierte el JSON de un experimento en un registro tabular."""
    record = {
        "experiment": experiment_dir.name,
        "result_file": str(experiment_dir / "final_results.json"),
        "task": data.get("task"),
        "backbone": data.get("backbone"),
        "strategy": data.get("strategy"),
        "test_f1_macro": data.get("test_f1_macro"),
        "test_balanced_acc": data.get("test_balanced_acc"),
        "test_auroc": data.get("test_auroc"),
        "best_val_f1": data.get("best_val_f1"),
        "timestamp": data.get("timestamp"),
    }

    for idx, value in enumerate(data.get("test_f1_per_class", [])):
        record[f"test_f1_class_{idx}"] = value

    confusion_matrix = data.get("test_confusion_matrix")
    if (
        isinstance(confusion_matrix, list)
        and len(confusion_matrix) == 2
        and all(isinstance(row, list) and len(row) == 2 for row in confusion_matrix)
    ):
        record.update(
            {
                "tn": confusion_matrix[0][0],
                "fp": confusion_matrix[0][1],
                "fn": confusion_matrix[1][0],
                "tp": confusion_matrix[1][1],
            }
        )

    return record


records = []
missing_files = []

for experiment_dir in sorted(RESULTS_DIR.iterdir()):
    if not experiment_dir.is_dir() or not experiment_dir.name.startswith("speech"):
        continue

    result_file = experiment_dir / "final_results.json"
    if not result_file.exists():
        missing_files.append(str(result_file))
        continue

    with result_file.open("r", encoding="utf-8") as fh:
        data = json.load(fh)

    records.append(flatten_final_results(data, experiment_dir))

if not records:
    raise ValueError("No se encontraron resultados en carpetas que empiecen por 'speech'.")

records = sorted(
    records,
    key=lambda row: (
        row.get("test_f1_macro") or 0,
        row.get("test_balanced_acc") or 0,
        row.get("test_auroc") or 0,
    ),
    reverse=True,
)

len(records), records[0]["experiment"]

(6, 'speech__efficientnet_b0__partial_ft')

In [3]:
preferred_columns = [
    "experiment",
    "task",
    "backbone",
    "strategy",
    "test_f1_macro",
    "test_balanced_acc",
    "test_auroc",
    "best_val_f1",
    "test_f1_class_0",
    "test_f1_class_1",
    "tn",
    "fp",
    "fn",
    "tp",
    "timestamp",
    "result_file",
]
extra_columns = sorted({key for row in records for key in row} - set(preferred_columns))
columns = [column for column in preferred_columns if any(column in row for row in records)] + extra_columns

output_json = RESULTS_DIR / "speech_results_consolidados.json"
output_csv = RESULTS_DIR / "speech_results_consolidados.csv"

with output_json.open("w", encoding="utf-8") as fh:
    json.dump(records, fh, indent=2, ensure_ascii=False)

with output_csv.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=columns)
    writer.writeheader()
    writer.writerows(records)

print(f"Resultados guardados en: {output_json}")
print(f"Resultados guardados en: {output_csv}")

if missing_files:
    print("Carpetas speech sin final_results.json:")
    for path in missing_files:
        print(f"- {path}")

Resultados guardados en: /home/odra/GitHub/ScraBrain/results/speech_results_consolidados.json
Resultados guardados en: /home/odra/GitHub/ScraBrain/results/speech_results_consolidados.csv
Carpetas speech sin final_results.json:
- /home/odra/GitHub/ScraBrain/results/speech__efficientnet_b0__partial_ft__source_lcmv/final_results.json


In [4]:
def format_value(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def green_gradient(value, min_value, max_value):
    if not isinstance(value, (int, float)) or min_value == max_value:
        return ""
    ratio = (value - min_value) / (max_value - min_value)
    ratio = max(0, min(1, ratio))
    start = (247, 252, 245)
    end = (49, 163, 84)
    rgb = tuple(round(start[i] + (end[i] - start[i]) * ratio) for i in range(3))
    text_color = "#ffffff" if ratio > 0.65 else "#111827"
    return f"background-color: rgb{rgb}; color: {text_color}; font-weight: 600;"


def render_table(rows, table_columns, gradient_columns=None):
    gradient_columns = set(gradient_columns or [])
    ranges = {}
    for column in gradient_columns:
        values = [row.get(column) for row in rows if isinstance(row.get(column), (int, float))]
        if values:
            ranges[column] = (min(values), max(values))

    header = "".join(f"<th>{html.escape(column)}</th>" for column in table_columns)
    body_rows = []
    for row in rows:
        cells = []
        for column in table_columns:
            value = row.get(column)
            style = ""
            if column in ranges:
                style = green_gradient(value, *ranges[column])
            cells.append(f"<td style='{style}'>{html.escape(format_value(value))}</td>")
        body_rows.append("<tr>" + "".join(cells) + "</tr>")
    body = "".join(body_rows)
    return HTML(
        """
        <style>
        .results-table { border-collapse: collapse; width: 100%; font-size: 13px; }
        .results-table th { background: #f3f4f6; text-align: left; }
        .results-table th, .results-table td { border: 1px solid #d1d5db; padding: 6px 8px; }
        </style>
        """
        + f"<table class='results-table'><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"
    )


summary_columns = [
    "experiment",
    "backbone",
    "strategy",
    "test_f1_macro",
    "test_balanced_acc",
    "test_auroc",
    "best_val_f1",
    "test_f1_class_0",
    "test_f1_class_1",
    "tn",
    "fp",
    "fn",
    "tp",
    "timestamp",
]
summary_columns = [column for column in summary_columns if column in columns]

gradient_columns = [
    "test_f1_macro",
    "test_balanced_acc",
    "test_auroc",
    "best_val_f1",
    "test_f1_class_0",
    "test_f1_class_1",
]

display(render_table(records, summary_columns, gradient_columns=gradient_columns))

experiment,backbone,strategy,test_f1_macro,test_balanced_acc,test_auroc,best_val_f1,test_f1_class_0,test_f1_class_1,tn,fp,fn,tp,timestamp
speech__efficientnet_b0__partial_ft,efficientnet_b0,partial_ft,0.6374,0.6480,0.6940,0.6305,0.4758,0.7990,374,339,485,1638,2026-04-29T18:35:02.390891
speech__resnet18__partial_ft,resnet18,partial_ft,0.6356,0.6609,0.7152,0.6340,0.4942,0.7769,429,284,594,1529,2026-04-29T17:01:51.785626
speech__vit_tiny__partial_ft,vit_tiny,partial_ft,0.6173,0.6362,0.6926,0.6331,0.4627,0.7720,391,322,586,1537,2026-04-29T20:17:28.324718
speech__efficientnet_b0__frozen,efficientnet_b0,frozen,0.6170,0.6128,0.6562,0.6127,0.4149,0.8190,278,435,349,1774,2026-04-29T17:41:53.316514
speech__resnet18__frozen,resnet18,frozen,0.6090,0.6096,0.6595,0.6120,0.4161,0.8019,299,414,425,1698,2026-04-29T16:36:59.642943
speech__vit_tiny__frozen,vit_tiny,frozen,0.6068,0.6061,0.6443,0.6095,0.4093,0.8042,289,424,410,1713,2026-04-29T19:26:02.782266


In [10]:
for row in records:
    display(Markdown(f"### {row['experiment']}"))

    metric_rows = [
        {"metrica": "Backbone", "valor": row.get("backbone")},
        {"metrica": "Estrategia", "valor": row.get("strategy")},
        {"metrica": "F1 macro test", "valor": row.get("test_f1_macro")},
        {"metrica": "Balanced accuracy test", "valor": row.get("test_balanced_acc")},
        {"metrica": "AUROC test", "valor": row.get("test_auroc")},
        {"metrica": "Mejor F1 validacion", "valor": row.get("best_val_f1")},
        {"metrica": "F1 clase 0", "valor": row.get("test_f1_class_0")},
        {"metrica": "F1 clase 1", "valor": row.get("test_f1_class_1")},
        {"metrica": "Timestamp", "valor": row.get("timestamp")},
    ]
    display(render_table(metric_rows, ["metrica", "valor"]))

    if {"tn", "fp", "fn", "tp"}.issubset(row):
        confusion_rows = [
            {"real": "Clase 0", "pred_clase_0": row["tn"], "pred_clase_1": row["fp"]},
            {"real": "Clase 1", "pred_clase_0": row["fn"], "pred_clase_1": row["tp"]},
        ]
        display(render_table(confusion_rows, ["real", "pred_clase_0", "pred_clase_1"]))


### speech__resnet18__partial_ft

metrica,valor
Backbone,resnet18
Estrategia,partial_ft
F1 macro test,0.6361
Balanced accuracy test,0.6281
AUROC test,0.6931
Mejor F1 validacion,0.6230
F1 clase 0,0.4363
F1 clase 1,0.8359
Timestamp,2026-04-28T11:34:29.690624


real,pred_clase_0,pred_clase_1
Clase 0,558,868
Clase 1,574,3672


### speech__efficientnet_b0__partial_ft

metrica,valor
Backbone,efficientnet_b0
Estrategia,partial_ft
F1 macro test,0.6313
Balanced accuracy test,0.6202
AUROC test,0.7052
Mejor F1 validacion,0.6343
F1 clase 0,0.4172
F1 clase 1,0.8454
Timestamp,2026-04-28T18:26:24.850263


...

metrica,valor
Backbone,vit_tiny
Estrategia,frozen
F1 macro test,0.5669
Balanced accuracy test,0.5641
AUROC test,0.6371
Mejor F1 validacion,0.5854
F1 clase 0,0.3044
F1 clase 1,0.8294
Timestamp,2026-04-28T12:34:06.842887


real,pred_clase_0,pred_clase_1
Clase 0,340,1086
Clase 1,468,3778


### speech__resnet18__frozen

metrica,valor
Backbone,resnet18
Estrategia,frozen
F1 macro test,0.4281
Balanced accuracy test,0.5000
AUROC test,0.6253
Mejor F1 validacion,0.4306
F1 clase 0,0.0000
F1 clase 1,0.8562
Timestamp,2026-04-28T16:25:25.555248


real,pred_clase_0,pred_clase_1
Clase 0,0,713
Clase 1,0,2123
